In [0]:
import json
from pyspark.sql import functions as F
dbutils.widgets.text("config", "", "Config record (JSON)")
print("change")



In [0]:
REQUIRED_KEYS = ["source_name", "source_folder", "ingestion_method", "source_format",
                 "is_active", "target_schema", "target_table", "load_type","target_catalog"]


def parse_config(raw: str) -> dict:
    """Parse the JSON config passed by the for-each task (object, or a single-item list)."""
    raw = (raw or "").strip()
    if not raw:
        raise ValueError("Parameter 'config' is empty. Pass one config record as JSON (for-each: {{input}}).")
    cfg = json.loads(raw)
    if isinstance(cfg, list):
        if len(cfg) != 1:
            raise ValueError(f"Expected one config record per run, got a list of {len(cfg)}. "
                             "Use this notebook inside a for-each task so each iteration gets one record.")
        cfg = cfg[0]
    missing = [k for k in REQUIRED_KEYS if cfg.get(k) in (None, "")]
    if missing:
        raise ValueError(f"Config is missing required keys: {missing}")

    # Normalise values
    cfg["ingestion_method"] = str(cfg["ingestion_method"]).strip().lower()
    cfg["source_format"] = str(cfg["source_format"]).strip().lower()
    cfg["load_type"] = str(cfg["load_type"]).strip().lower()
    cfg["is_active"] = cfg["is_active"] if isinstance(cfg["is_active"], bool) \
        else str(cfg["is_active"]).strip().lower() in ("true", "1", "yes")

    if cfg["source_format"] not in ("csv", "tsv", "json", "parquet"):
        raise ValueError(f"Unsupported source_format: {cfg['source_format']}")
    if cfg["load_type"] not in ("incremental", "overwrite"):
        raise ValueError(f"Unsupported load_type: {cfg['load_type']}")
    return cfg


cfg = parse_config(dbutils.widgets.get("config"))
catalog = cfg["target_catalog"]
checkpoint_base = cfg["checkpoint_base"].rstrip("/")

target_table = f"`{catalog}`.`{cfg['target_schema']}`.`{cfg['target_table']}`"
checkpoint_path = f"{checkpoint_base}/{cfg['target_schema']}/{cfg['target_table']}"
schema_location = f"{checkpoint_path}/_schema"

print(json.dumps(cfg, indent=2, default=str))
print("Target table   :", target_table)
print("Checkpoint path:", checkpoint_path)

In [0]:
schema_exists = spark.sql(
    f"SELECT 1 FROM `{catalog}`.information_schema.schemata WHERE schema_name = '{cfg['target_schema']}'"
).count() > 0
if not schema_exists:
    raise ValueError(f"Target schema `{catalog}`.`{cfg['target_schema']}` does not exist. "
                     "Create it or fix target_schema in the config table.")
parts = checkpoint_base.split("/")
if checkpoint_base.startswith("/Volumes/") and len(parts) >= 5:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{parts[2]}`.`{parts[3]}`.`{parts[4]}`")

FORMAT_OPTIONS = {
    "csv": {"header": "true"},
    "tsv": {"header": "true", "sep": "\t"},
    "json": {"multiLine": "false"},
    "parquet": {},
}

# Spark has no 'tsv' format: TSV is read as CSV with a tab separator
SPARK_FORMAT = {"tsv": "csv"}
spark_format = SPARK_FORMAT.get(cfg["source_format"], cfg["source_format"])


import re

# Characters Delta does not allow in column names (without column mapping)
INVALID_COLUMN_CHARS = r"[ ,;{}()\n\t=]"


def clean_column_names(df):
    """Replace spaces and other invalid characters in column names with '_'."""
    return df.select([F.col(f"`{c}`").alias(re.sub(INVALID_COLUMN_CHARS, "_", c.strip())) for c in df.columns])


def add_audit_columns(df):
    # _metadata must be read before any rename/select, then clean the column names
    df = df.select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.current_timestamp().alias("_ingested_at"),
        F.lit(cfg.get("config_id")).cast("bigint").alias("_config_id"),
    )
    return clean_column_names(df)


def ingest_autoloader_incremental() -> int:
    """Auto Loader: picks up only new files since the last run, appends, then stops (availableNow)."""
    reader = (spark.readStream.format("cloudFiles")
              .option("cloudFiles.format", spark_format)
              .option("cloudFiles.schemaLocation", schema_location)
              .option("cloudFiles.schemaEvolutionMode", "addNewColumns"))
    if cfg["source_format"] in ("csv", "json","tsv"):
        reader = reader.option("cloudFiles.inferColumnTypes", "true")
    df = add_audit_columns(reader.options(**FORMAT_OPTIONS[cfg["source_format"]]).load(cfg["source_folder"]))

    query = (df.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", checkpoint_path)
             .option("mergeSchema", "true")
             .trigger(availableNow=True)
             .toTable(target_table))
    query.awaitTermination()


def ingest_full_overwrite() -> int:
    """Batch full reload: reads every file in the folder and replaces the target table."""
    reader = spark.read.format(spark_format).options(**FORMAT_OPTIONS[cfg["source_format"]])
    if spark_format == "csv":
        reader = reader.option("inferSchema", "true")
    df = add_audit_columns(reader.load(cfg["source_folder"]))
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(target_table))
    return spark.table(target_table).count()

In [0]:
method, load_type = cfg["ingestion_method"], cfg["load_type"]

if method == "autoloader":
    rows = ingest_autoloader_incremental() if load_type == "incremental" else ingest_full_overwrite()
elif method in ("lakeflow_pipeline", "declarative_pipeline"):
    # Pipelines are run by a pipeline task in the job, not from this notebook
    raise NotImplementedError(f"ingestion_method '{method}' should be run as a pipeline task, not by this notebook.")
else:
    raise ValueError(f"Unknown ingestion_method: {method}")

result = {"config_id": cfg.get("config_id"), "target_table": target_table, "load_type": load_type,
          "rows_processed": rows, "status": "SUCCESS"}
print(result)
dbutils.notebook.exit(json.dumps(result))